In [1]:
## Basic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast


## For Preproc
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('stopwords')
nltk.download('wordnet')

## Vectorization of Text
from sklearn.feature_extraction.text import TfidfVectorizer

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AnuragS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\AnuragS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
import os
os.chdir(r'C:\Users\AnuragS\OneDrive\cv_project\Movie Recommendation System\data')
os.getcwd()

'C:\\Users\\AnuragS\\OneDrive\\cv_project\\Movie Recommendation System\\data'

In [3]:
df = pd.read_csv('cleaned_data_for_anylytics.csv')
df.head()

,id,title,release_year,vote_average,popularity,tag
0,19995,Avatar,2009,7.2,150.437577,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,2007,6.9,139.082615,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,2015,6.3,107.376788,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,2012,7.6,112.312950,following the death of district attorney harve...
4,49529,John Carter,2012,6.1,43.926995,"john carter is a war-weary, former military ca..."


In [4]:
print(df.iloc[0]['tag'])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron


In [5]:
print(df.iloc[1]['tag'])

captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. adventure fantasy action captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but nothing is quite as it seems. ocean drugabuse exoticisland eastindiatradingcompany loveofone'slife traitor shipwreck strongwoman ship alliance calypso afterlife fighter pirate swashbuckler aftercreditsstinger johnnydepp orlandobloom keiraknightley goreverbinski


In [6]:
obj_stop_words = set(stopwords.words('english'))
obj_lemmatizer =  WordNetLemmatizer()

In [7]:
def f_preproc_text(text):
    """
    1. lowercase text
    2. remove punctuation
    3. remove stopwords
    4. Lemmatize
    """
    text = str(text).lower() #1
    text = re.sub(r'[^\w\s]', '', text) #2
    words = text.split() #3
    words = [w for w in words if w not in obj_stop_words] #3
    words = [obj_lemmatizer.lemmatize(w1) for w1 in words] #4
    word_string = ' '.join(words) # make the string
    return word_string

In [8]:
print('Old Tag:')
print(df.iloc[0]['tag'])
print(' ')
print('Preproc Tag:')
print(f_preproc_text(df.iloc[0]['tag']))

Old Tag:
in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron
 
Preproc Tag:
22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following order protecting alien civilization action adventure fantasy sciencefiction 22nd century paraplegic marine dispatched moon pandora unique mission becomes torn following order protecting alien civilization cultureclash future spacewar spacecolony socie

In [13]:
print('Old Tag:')
print(df.iloc[1234]['tag'])
print(' ')
print('Preproc Tag:')
print(f_preproc_text(df.iloc[1234]['tag']))

Old Tag:
ruthless terrorist threaten bring united nation frame one man believe stop international security expert named shaw must run ally become solitary force good try stop could become world war iii crime action adventure ruthless terrorist threaten bring united nation frame one man believe stop international security expert named shaw must run ally become solitary force good try stop could become world war iii china chinesewoman secretagent conspiracyofmurder unitednations wesleysnipes donaldsutherland maurychaykin christianduguay
 
Preproc Tag:
ruthless terrorist threaten bring united nation frame one man believe stop international security expert named shaw must run ally become solitary force good try stop could become world war iii crime action adventure ruthless terrorist threaten bring united nation frame one man believe stop international security expert named shaw must run ally become solitary force good try stop could become world war iii china chinesewoman secretagent cons

In [9]:
df['tag'] = df['tag'].apply(f_preproc_text)

In [14]:
df = df.reset_index(drop=True)

In [15]:
df.head()

,id,title,release_year,vote_average,popularity,tag
0,19995,Avatar,2009,7.2,150.437577,22nd century paraplegic marine dispatched moon...
1,285,Pirates of the Caribbean: At World's End,2007,6.9,139.082615,captain barbossa long believed dead come back ...
2,206647,Spectre,2015,6.3,107.376788,cryptic message bond past sends trail uncover ...
3,49026,The Dark Knight Rises,2012,7.6,112.312950,following death district attorney harvey dent ...
4,49529,John Carter,2012,6.1,43.926995,john carter warweary former military captain w...


In [16]:
indices = pd.Series(df.index,index = df['title']).drop_duplicates()
indices

title
Avatar                                         0
Pirates of the Caribbean: At World's End       1
Spectre                                        2
The Dark Knight Rises                          3
John Carter                                    4
                                            ... 
El Mariachi                                 4794
Newlyweds                                   4795
Signed, Sealed, Delivered                   4796
Shanghai Calling                            4797
My Date with Drew                           4798
Length: 4799, dtype: int64

1. OHE Method can lead to inconsistency because of the size
2. Bagofwords - only 1 and 0

We will use TFIDF

In [29]:
indices['Signed, Sealed, Delivered']

np.int64(4796)

In [47]:
tfidf = TfidfVectorizer(stop_words='english',
                        max_features=5000, ## vocab size is limited to the number of movies
                        ngram_range=(1,2) 
                       )

In [48]:
tfidf_matrix = tfidf.fit_transform(df['tag'])

In [49]:
tfidf_matrix[123].shape

(1, 5000)

## Cosin similarity

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
indexy = indices['Signed, Sealed, Delivered']
#indexy
smiy = cosine_similarity(tfidf_matrix[indexy], tfidf_matrix).flatten()
smiy_idx =  smiy.argsort()[::-1][1:6]
df['title'].iloc[smiy_idx]

1591         Letters to Juliet
1043         Miss Congeniality
2694    I Think I Love My Wife
4033            Christmas Mail
1083              Babylon A.D.
Name: title, dtype: object

In [63]:
def f_recommend(inp_title,n=10):
    """
    This function will help to recommend movies
    """
    if inp_title not in indices:
        return ['Movie not found']

    index = indices[inp_title]
    similarity_score = cosine_similarity(tfidf_matrix[index],tfidf_matrix).flatten()
    similar_index = similarity_score.argsort()[::-1][1:n+1]
    df_recom = df['title'].iloc[similar_index]
    return df_recom

In [66]:
f_recommend('Babylon A.D.',4)

2258              Steamboy
1664         Dead Man Down
4565        Mutual Friends
1515    Laws of Attraction
Name: title, dtype: object

In [67]:
import joblib

os.makedirs("../model",exist_ok=True)

joblib.dump(tfidf_matrix,"../model/tfidf_matrix.pkl")
joblib.dump(indices,"../model/indices.pkl")
joblib.dump(df,"../model/df.pkl")
joblib.dump(tfidf,"../model/tfidf.pkl")

['../model/tfidf.pkl']